# Character-level language modeling

## Step 1 Preprocessing

In [8]:
from matplotlib import backend_bases
import numpy as np

with open('1268-0.txt', 'r', encoding = 'utf-8') as fp:
    text = fp.read()

start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find("End of the Project Gutenberg")

text = text[start_indx:end_indx]
char_set = set(text)
vocab_size = len(char_set)

print("Total len ", len(text))
print("Unique characters ",vocab_size)

Total len  1112310
Unique characters  80


Converting characters to no

In [9]:
sorted_char = np.array(sorted(char_set))
char2int = { ch:i for i , ch in enumerate(sorted_char)}

# text = "Hello"
# encoded_text = [ char2int[ch] for ch in text ]
# print(f"Encoded Text {encoded_text}")
# decoded_text = sorted_char[encoded_text]
# print(f"Decoded Text {decoded_text}")

context_len = 40
length = end_indx - context_len + 1

# for i in range(len(text[:100])):
#     print(f"Input {text[i : i + context_len]} , output {text[i+ context_len]}" )

text_chunks = [text[i : i + context_len] for i in range(len(text) - context_len)]
print(text_chunks[0])

text_encoded = [ [char2int[ch] for ch in chunk] for chunk in text_chunks]
print(text_encoded[0])

THE MYSTERIOUS ISLAND

by Jules Verne

1
[44, 32, 29, 1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43, 1, 33, 43, 36, 25, 38, 28, 0, 0, 51, 74, 1, 34, 70, 61, 54, 68, 1, 46, 54, 67, 63, 54, 0, 0, 12]


Prepare the data using pytorch Dataset class

In [10]:
from torch.cuda._sanitizer import enable_cuda_sanitizer
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        return self.text_chunks[idx][:-1] , self.text_chunks[idx][1:]

seq_dataset = TextDataset(torch.tensor(text_encoded))

seq_dataset
for i , (seq,target) in enumerate(seq_dataset):
    print(i)
    print(f"Input {seq} : ouput {target}")

    if i == 1:
        break

0
Input tensor([44, 32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36,
        25, 38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63,
        54,  0,  0]) : ouput tensor([32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36, 25,
        38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63, 54,
         0,  0, 12])
1
Input tensor([32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36, 25,
        38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63, 54,
         0,  0, 12]) : ouput tensor([29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36, 25, 38,
        28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63, 54,  0,
         0, 12, 19])


## Step 2 Model Training

Define the model

In [11]:
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

# Step 1: Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

batch_size = 64
torch.manual_seed(1)

seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

class RNN(nn.Module):
    def __init__(self, embed_size, rnn_hid_size, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.hidden_size = rnn_hid_size
        self.rnn = nn.LSTM(embed_size, rnn_hid_size, batch_first=True)
        self.fc = nn.Linear(rnn_hid_size, vocab_size)

    def forward(self, x):
        output = self.embedding(x)
        output, (hidden, cell) = self.rnn(output)
        output = self.fc(output)
        return output, hidden, cell

# Step 2: Move model to device
model = RNN(256, 512, vocab_size).to(device)
model

Using device: cuda


RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

Define the loss function and optimers

In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Training

In [13]:
epochs = 20

for epoch in range(epochs):
    total_loss = 0
    for x, y in seq_dl:
        # Step 3: Move data to device
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, _, _ = model(x)

        loss = loss_fn(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch}, Loss: {total_loss/len(seq_dl)}')

Epoch 0, Loss: 1.0921095072780684
Epoch 1, Loss: 0.8887974771354983
Epoch 2, Loss: 0.8245464009714316
Epoch 3, Loss: 0.7938995942592594
Epoch 4, Loss: 0.7761914962982512
Epoch 5, Loss: 0.764180995304858
Epoch 6, Loss: 0.7555120950793567
Epoch 7, Loss: 0.749103753951142
Epoch 8, Loss: 0.7440752369761159
Epoch 9, Loss: 0.7400883013904832
Epoch 10, Loss: 0.73695071934419
Epoch 11, Loss: 0.7339254818396709
Epoch 12, Loss: 0.7317734788446268
Epoch 13, Loss: 0.7300187457732072
Epoch 14, Loss: 0.7286786234785576
Epoch 15, Loss: 0.7272482359256368
Epoch 16, Loss: 0.7268732519588507
Epoch 17, Loss: 0.7260215417535816
Epoch 18, Loss: 0.7259218282451001
Epoch 19, Loss: 0.7259682098708923


In [14]:
model.eval()
seed = "THE"

# Step 4: Move inference data to device
input_ids = torch.tensor(
    [[char2int[c] for c in seed]]
).to(device)

generated = seed

with torch.no_grad():
    for _ in range(300):
        logits, hidden, cell = model(input_ids)

        probs = torch.softmax(
            logits[:, -1],
            dim=-1
        )

        idx = torch.multinomial(
            probs,
            1
        )

        generated += sorted_char[idx.item()]

        # Concatenate on the same device
        input_ids = torch.cat(
            [input_ids, idx],
            dim=1
        )

        input_ids = input_ids[:, -100:]

print(generated)

THE, Thosembled them again and should
then be discontinued. During the five streams of wood force very threatening therefore
confident, this was all that had happened, or anger. “It is more
perhaps!”

“Hallo, That case, captain!” said Herbert; “their more worked motionless
and rough, for it was in good


# Step 3 Prediction